# Privacy Metric Example

The **Privacy** metric scores how suitable a PII/PHI detector is for your domain, and `PrivacyRanker` orders several detectors so you can pick one. The score combines five factors:

```
Score = DetectionScore × Coverage × DomainFit × RegulatoryFit × Penalty_FN
```

plus a complementary **risk index** (`R_final`) that flags critical blind spots. This notebook runs with **no extra dependencies** — it uses a tiny regex detector so you can see the full input → output shape. Swap in `PresidioDetector` / `HuggingFacePIIDetector` once you install the corresponding extra.

## Installation

In [ ]:
# Metric core needs no extras. For real backends use one of:
#   !pip install "gaussia[privacy-presidio]" matplotlib -q
#   !pip install "gaussia[privacy-huggingface]" matplotlib -q
!pip install "gaussia" matplotlib -q

## Setup

Three pieces: a **domain config** (which classes matter + their weights), one or more **detectors**, and a **retriever** that yields labelled `PrivacyBatch`es (`query` = text shown to the detector, `spans` = ground truth).

In [ ]:
import re

from gaussia.core.detector import PIIDetector
from gaussia.core.retriever import Retriever
from gaussia.metrics.privacy import Privacy, PrivacyRanker
from gaussia.schemas.common import Dataset
from gaussia.schemas.privacy import PrivacyBatch, PrivacyDomainConfig, Span

# The evaluation domain. Both weight maps must sum to 1.0.
#   criticality_weights -> how much each class counts toward the DetectionScore
#   fn_severity_weights -> how costly a MISS of each class is (drives the risk index)
DOMAIN = PrivacyDomainConfig(
    classes=frozenset({"email_address", "phone_number"}),
    criticality_weights={"email_address": 0.6, "phone_number": 0.4},
    fn_severity_weights={"email_address": 0.7, "phone_number": 0.3},
    iou_threshold=0.5,
    regulatory_framework="GDPR",
)

_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
_PHONE = re.compile(r"\d{3}-\d{4}")


class RegexDetector(PIIDetector):
    """A trivial detector so the notebook runs without optional backends."""

    @property
    def supported_classes(self) -> frozenset[str]:
        return frozenset({"email_address", "phone_number"})

    def predict(self, text: str) -> list[Span]:
        spans = [Span(label="email_address", start=m.start(), end=m.end(), text=m.group(), score=0.99) for m in _EMAIL.finditer(text)]
        spans += [Span(label="phone_number", start=m.start(), end=m.end(), text=m.group(), score=0.90) for m in _PHONE.finditer(text)]
        return spans


class EmailOnlyDetector(PIIDetector):
    """A weaker detector: only finds emails -> lower coverage and a phone blind spot."""

    @property
    def supported_classes(self) -> frozenset[str]:
        return frozenset({"email_address"})

    def predict(self, text: str) -> list[Span]:
        return [Span(label="email_address", start=m.start(), end=m.end(), text=m.group(), score=0.99) for m in _EMAIL.finditer(text)]


class InMemoryRetriever(Retriever):
    """Yields one labelled conversation. Replace with your own Retriever subclass."""

    def load_dataset(self) -> list[Dataset]:
        text = "Reach me at john@example.com or 555-1234."
        turn = PrivacyBatch(
            qa_id="turn-1",
            query=text,
            assistant="",
            ground_truth_assistant="",
            spans=[
                Span(label="email_address", start=12, end=28, text="john@example.com"),
                Span(label="phone_number", start=32, end=40, text="555-1234"),
            ],
        )
        return [Dataset(session_id="demo", assistant_id="bot", context="", conversation=[turn])]

## Evaluate a single detector

`Privacy.run` returns one `PrivacyMetric` per dataset. To use a real backend, replace the detector with e.g. `PresidioDetector(name="presidio", domain_fit=0.9, regulatory_fit=0.8)` after installing `gaussia[privacy-presidio]`.

In [ ]:
detector = RegexDetector(name="regex-baseline", domain_fit=0.9, regulatory_fit=0.8)
metric = Privacy.run(InMemoryRetriever, detector=detector, domain_config=DOMAIN)[0]

print(f"detector        : {metric.name}")
print(f"score (0-100)   : {metric.score_100:.2f}  -> {metric.interpretation}")
print(f"detection_score : {metric.detection_score:.3f}")
print(f"coverage        : {metric.coverage:.3f}")
print(f"penalty_fn      : {metric.penalty_fn:.3f}")
print(f"risk (r_final)  : {metric.r_final:.3f}  (weakest class: {metric.r1_weakest_class})")
print("\nper-class breakdown:")
for label, cm in metric.class_metrics.items():
    print(f"  {label:<14} f2={cm.f2:.2f}  tp={cm.tp} fp={cm.fp} fn={cm.fn}")

## Rank several detectors

`PrivacyRanker.run` evaluates every detector against the same corpus and returns a `PrivacyRanking` ordered by `score_100` (failures sorted to the tail; if a detector raises it is recorded with `success=False` and the ranking still completes).

In [ ]:
good = RegexDetector(name="regex-baseline", domain_fit=0.9, regulatory_fit=0.8)
weak = EmailOnlyDetector(name="email-only", domain_fit=0.5, regulatory_fit=0.5)

ranking = PrivacyRanker.run(InMemoryRetriever, detectors=[good, weak], domain_config=DOMAIN)[0]

print(f"winner: {ranking.winning_detector}\n")
for rank, result in enumerate(ranking.results, start=1):
    print(f"  {rank}. {result.name:<16} score_100={result.score_100:6.2f}  success={result.success}")

## Visualize the ranking

In [ ]:
import matplotlib.pyplot as plt

names = [r.name for r in ranking.results]
scores = [r.score_100 for r in ranking.results]

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(names[::-1], scores[::-1], color="#4C72B0")
ax.set_xlabel("score_100")
ax.set_xlim(0, 100)
ax.set_title("Detector ranking (higher is better)")
for i, s in enumerate(scores[::-1]):
    ax.text(s + 1, i, f"{s:.1f}", va="center")
plt.tight_layout()
plt.show()

## Understanding the results

- **`score_100`** is the headline number; **`interpretation`** maps it to a qualitative band (`"Not suitable"` … `"Recommended with strong local evidence"`).
- The score is the **product** of five factors, so a single weak factor (e.g. a detector that doesn't *support* a class → low `coverage`, or misses a critical class → low `penalty_fn`) drags the whole score down. That is intentional.
- **`r_final`** is the risk index: it stays high when a detector has a critical blind spot even if its average score looks acceptable — inspect `r1_weakest_class` to see which class.
- **`load_time` / `inference_latency`** are reported for diagnostics only and never enter the score.
- To plug in your own model, subclass `PIIDetector` (implement `predict` and `supported_classes`) — no library change needed, and it slots straight into `Privacy` / `PrivacyRanker`.

---
# Part 2 — A more realistic walkthrough

Part 1 was a toy. Here we build something closer to real life so you can see **exactly what you, the user, provide** and **how the metric reacts to different kinds of failure**.

The three pieces you always supply are:
1. **the domain** — which PII classes matter and how much (`PrivacyDomainConfig`),
2. **the detectors** — the models you want to compare (each a `PIIDetector`),
3. **the dataset** — your labelled corpus, as `PrivacyBatch`es behind a `Retriever`.

Everything else (the score math, the risk index, the ranking) is the library's job.


## 2.1 — The domain you care about

We evaluate a banking/health chatbot over **4 classes**. The weights are *your* policy decision:
- `criticality_weights` (w): how much each class counts **toward the score** when detected well.
- `fn_severity_weights` (rho): how much it **hurts to MISS** each class (drives the risk index).

Here `credit_card` is the most critical to miss (rho=0.45): losing a card number is the worst outcome. Both maps must sum to 1.0.


In [ ]:
from gaussia.core.detector import PIIDetector
from gaussia.core.retriever import Retriever
from gaussia.metrics.privacy import Privacy, PrivacyRanker
from gaussia.schemas.common import Dataset
from gaussia.schemas.privacy import PrivacyBatch, PrivacyDomainConfig, Span

DOMAIN = PrivacyDomainConfig(
    classes=frozenset({"person", "email_address", "credit_card", "us_ssn"}),
    # w: importance for the DetectionScore (must sum to 1.0)
    criticality_weights={"person": 0.20, "email_address": 0.20, "credit_card": 0.35, "us_ssn": 0.25},
    # rho: how costly a MISS is, per class (must sum to 1.0). credit_card hurts most.
    fn_severity_weights={"person": 0.15, "email_address": 0.15, "credit_card": 0.45, "us_ssn": 0.25},
    iou_threshold=0.5,           # a predicted span counts as a hit if it overlaps >=50% of the true span
    regulatory_framework="PCI-DSS",
)
print("domain classes:", sorted(DOMAIN.classes))


## 2.2 — Your labelled dataset

This is the part **you** own: your texts with the **true** PII marked. Each turn is a `PrivacyBatch`:
- `query` = the text the detector sees,
- `spans` = the ground-truth PII (label + character offsets `[start, end)`).

Instead of hardcoding offsets by hand, the `gt(...)` helper finds each value inside the text — that is exactly the kind of small parser you would write in your own `Retriever`. Note turn 3 has **no PII** (`spans=[]`): that is allowed and important (it lets false positives show up).


In [ ]:
def gt(query: str, *items: tuple[str, str]) -> list[Span]:
    """Build ground-truth spans by locating each (label, value) inside the query."""
    spans = []
    for label, value in items:
        start = query.index(value)
        spans.append(Span(label=label, start=start, end=start + len(value), text=value))
    return spans

TURNS = [
    ("t1", "Hi, I'm John Doe, email john@acme.com",
        gt("Hi, I'm John Doe, email john@acme.com",
           ("person", "John Doe"), ("email_address", "john@acme.com"))),
    ("t2", "My card is 4111-1111-1111-1111 and SSN 123-45-6789",
        gt("My card is 4111-1111-1111-1111 and SSN 123-45-6789",
           ("credit_card", "4111-1111-1111-1111"), ("us_ssn", "123-45-6789"))),
    ("t3", "Thanks, that's all!", []),   # no PII in this turn
]


class CorpusRetriever(Retriever):
    """Turns your data into the Gaussia format. This is the only 'glue' you write."""

    def load_dataset(self) -> list[Dataset]:
        conversation = [
            PrivacyBatch(qa_id=qid, query=text, assistant="", ground_truth_assistant="", spans=spans)
            for qid, text, spans in TURNS
        ]
        return [Dataset(session_id="chat-001", assistant_id="bank-bot", context="", conversation=conversation)]

total_pii = sum(len(s) for _, _, s in TURNS)
print(f"{len(TURNS)} turns, {total_pii} ground-truth PII spans")


## 2.3 — The detectors (models) under test

In real use these are `PresidioDetector`, `HuggingFacePIIDetector`, or your own subclass. To make the lesson deterministic we use a small `FixedDetector` whose predictions are spelled out, so you can predict the scores yourself. We compare **three** detectors:

- **`comprehensive`** — finds everything correctly. Should score highest, zero risk.
- **`no-financial`** — good, but **never detects credit_card** (the most critical class). Watch the **risk index spike** even though it gets 3/4 classes.
- **`noisy`** — only supports person/email (low coverage), emits a **false positive**, an **out-of-domain** prediction, and an **overlapping duplicate** (dropped by NMS).


In [ ]:
class FixedDetector(PIIDetector):
    """Deterministic detector: returns the predictions we hand it, per text."""
    classes_supported: frozenset[str]
    predictions: dict[str, list[Span]] = {}

    @property
    def supported_classes(self) -> frozenset[str]:
        return self.classes_supported

    def predict(self, text: str) -> list[Span]:
        return self.predictions.get(text, [])


def pred(query: str, label: str, value: str, score: float) -> Span:
    """A prediction span located by substring (value need not be a true PII)."""
    start = query.index(value)
    return Span(label=label, start=start, end=start + len(value), text=value, score=score)

t1 = "Hi, I'm John Doe, email john@acme.com"
t2 = "My card is 4111-1111-1111-1111 and SSN 123-45-6789"

# 1) comprehensive: detects every true PII, supports all 4 classes
comprehensive = FixedDetector(
    name="comprehensive", domain_fit=0.9, regulatory_fit=0.85,
    classes_supported=frozenset({"person", "email_address", "credit_card", "us_ssn"}),
    predictions={
        t1: [pred(t1, "person", "John Doe", 0.99), pred(t1, "email_address", "john@acme.com", 0.99)],
        t2: [pred(t2, "credit_card", "4111-1111-1111-1111", 0.97), pred(t2, "us_ssn", "123-45-6789", 0.96)],
    },
)

# 2) no-financial: supports all 4 but NEVER returns credit_card -> critical blind spot
no_financial = FixedDetector(
    name="no-financial", domain_fit=0.8, regulatory_fit=0.9,
    classes_supported=frozenset({"person", "email_address", "credit_card", "us_ssn"}),
    predictions={
        t1: [pred(t1, "person", "John Doe", 0.95), pred(t1, "email_address", "john@acme.com", 0.95)],
        t2: [pred(t2, "us_ssn", "123-45-6789", 0.9)],   # credit_card MISSED
    },
)

# 3) noisy: supports only person/email; adds a false positive, an out-of-domain label, and an overlap
noisy = FixedDetector(
    name="noisy", domain_fit=0.6, regulatory_fit=0.6,
    classes_supported=frozenset({"person", "email_address"}),
    predictions={
        t1: [
            pred(t1, "person", "John Doe", 0.9),       # correct -> TP
            pred(t1, "person", "John D", 0.4),         # overlaps the above -> dropped by NMS
            pred(t1, "person", "email", 0.8),          # wrong -> false positive
            pred(t1, "email_address", "john@acme.com", 0.9),  # correct -> TP
            pred(t1, "ip_address", "Hi", 0.7),         # label NOT in domain -> filtered (not a FP)
        ],
        t2: [],  # supports neither credit_card nor us_ssn -> both missed
    },
)
print("3 detectors ready")


## 2.4 — Rank them

One call. The ranker evaluates each detector over the corpus and orders them by `score_100`.


In [ ]:
ranking = PrivacyRanker.run(
    CorpusRetriever,
    detectors=[comprehensive, no_financial, noisy],
    domain_config=DOMAIN,
)[0]

print(f"WINNER: {ranking.winning_detector}\n")
header = f"{'detector':<16}{'score':>8}{'detect':>8}{'cover':>8}{'penalty':>9}{'risk':>8}{'OOD':>5}"
print(header); print("-" * len(header))
for r in ranking.results:
    print(f"{r.name:<16}{r.score_100:>8.2f}{r.detection_score:>8.2f}{r.coverage:>8.2f}"
          f"{r.penalty_fn:>9.2f}{r.r_final:>8.2f}{r.out_of_domain_filtered:>5}")


## 2.5 — Read the result, detector by detector

Inspect the per-class breakdown to see *why* each detector scored what it did.


In [ ]:
for r in ranking.results:
    print(f"\n=== {r.name}  (score_100={r.score_100:.2f} -> {r.interpretation}) ===")
    print(f"    risk r_final={r.r_final:.2f}  weakest class={r.r1_weakest_class}  "
          f"out_of_domain_filtered={r.out_of_domain_filtered}")
    for label in sorted(DOMAIN.classes):
        cm = r.class_metrics[label]
        print(f"    {label:<14} f2={cm.f2:.2f}  precision={cm.precision:.2f} recall={cm.recall:.2f}"
              f"  (tp={cm.tp} fp={cm.fp} fn={cm.fn})")


## 2.6 — What you should see, and why

| detector | score | what happened |
|---|---|---|
| **comprehensive** | ~**76.5** | finds every PII (all f2=1) -> detection=1.0, coverage=1.0, penalty=1.0. Score = 0.9x0.85 (just the fit scalars). `r_final=0`. |
| **no-financial** | ~**25.7** | gets person/email/ssn perfectly, but **misses every credit_card**. Because credit_card has the highest miss-severity (rho=0.45), `penalty_fn` drops to 0.55 and **`r_final` jumps to ~0.70** with `weakest class=credit_card`. The score collapses even though it got 3/4 classes. |
| **noisy** | ~**2.0** | a false positive drags `person` precision to 0.5; it only **supports 2/4 classes** (coverage 0.5); it misses both financial classes (penalty 0.30). The bogus `ip_address` prediction is **filtered** (`out_of_domain_filtered=1`) and the overlapping `John D` is dropped by NMS. Many weak factors multiply -> tiny score. |

**The big lesson:** the score is a **product of five factors**, so one bad dimension sinks it. And the **risk index is not the score** — `no-financial` looks okay on raw detection but its risk is high because of *what* it misses. Always read both.

To use this for real: keep the same structure, point `CorpusRetriever` at *your* labelled data, and pass real detectors (`PresidioDetector(...)`, `HuggingFacePIIDetector(model_path=...)`, or your own `PIIDetector` subclass).
